# 2.1 Kernels: what a kernel scans, fewer positions, and how far one position sees

In lesson 1 a frozen encoder did the heavy work: it had already ordered the vectors, and
you trained the cheap part on top. Here there is no encoder. A heartbeat is 187 numbers,
and whatever structure the model uses, it has to find in those numbers itself.

This notebook gives you the first block for that: the **kernel**. It works the way
lesson 1 worked: a baseline with its spread first, then one bet at a time, each with
what you expect written down before the cell runs.

| section | the question | goal |
|---|---|---|
| 2.1.1 | what does a model without kernels reach on unseen patients? | 1.13, 2.9 |
| 2.1.2 | what does a kernel do, and why is it one operation at every rank? | 2.2, 1.6 |
| 2.1.3 | how do you get fewer positions? | 2.3 |
| 2.1.4 | how far does one position see? | 2.4 |
| 2.1.5 | does a wider view pay on heartbeats? | 2.4, 2.7 |
| 2.1.6 | the same beat as a 2D grid | 2.8 |

In [ ]:
import sys

import heartbeats as hb
import matplotlib.pyplot as plt
import numpy as np
import torch
from loguru import logger
from notebooktester import param
from sklearn.neighbors import KNeighborsClassifier
from torch import nn
from vectormesh.components import Conv1D, Conv2D, Pool1D, Pool2D

logger.remove()
logger.add(sys.stderr, level="WARNING")

EPOCHS = param(8, test=1)
REPEATS = param(3, test=1)
MAX_ROWS = param(None, test=2000)  # fit rows; None keeps all of them

fit, val = hb.load(max_rows=MAX_ROWS)
print(f"fit  {len(fit):6d} beats  {fit.counts()}")
print(f"val  {len(val):6d} beats  {val.counts()}  records {hb.VAL_RECORDS}")
print(f"one batch item: {tuple(fit.beats[0].shape)}  (S, D): 187 steps, one channel")

## The data, and why it is split by patient

MIT-BIH holds 48 half-hour recordings of 47 patients. The cache is split **by patient**:
22 patients train, 22 others test, and here 6 of the train patients are held out as
validation. A beat of a patient the model trained on is not a new case: the next beat of
the same heart looks almost the same. Split per beat instead, and every score is higher
than the model would reach on the next patient (goal 1.9).

Every beat starts at its R-peak, the tall spike, and runs for 1.2 times the time between
two beats of that stretch of the recording; the rest is zeros. Three classes:

- **N**, normal and other non-ectopic beats;
- **S**, supraventricular ectopic: a beat that starts too early, above the ventricles;
- **V**, ventricular ectopic: a beat that starts in the ventricles, wide and odd-shaped.

In [ ]:
rng = np.random.default_rng(0)
labels, records = fit.labels.numpy(), fit.records
fig, axes = plt.subplots(2, 3, figsize=(12, 5.5), sharex=True, sharey=True)
t = np.arange(hb.STEPS) / 125
for k, name in enumerate(hb.CLASSES):
    of_class = np.flatnonzero(labels == k)
    most = np.bincount(records[of_class]).argmax()  # the patient with most beats of k
    mine = of_class[records[of_class] == most]
    one = rng.choice(mine, min(20, len(mine)), replace=False)
    many = rng.choice(of_class, min(20, len(of_class)), replace=False)
    for row, rows, who in ((0, one, "one patient"), (1, many, None)):
        for r in rows:
            axes[row, k].plot(t, fit.beats[r, :, 0], color="C0", alpha=0.25, lw=1)
        who = who or f"{len(set(records[rows]))} patients"
        axes[row, k].set_title(f"{name}: {len(rows)} beats of {who}")
    axes[1, k].set_xlabel("seconds after the R-peak")
for ax in axes[:, 0]:
    ax.set_ylabel("scaled amplitude")
plt.tight_layout()
plt.show()

### Look before you design

Every design below is a bet about these pictures (slide 17). The top row is one patient
per class, the bottom row many. Within one heart the beats of a class lie on top of each
other; across hearts they spread. That spread is why the data is split by patient, and
it is what a model has to see through. Before any model, say which class you would trust
a model to find, and from what you see:

- in **N**, a second spike near the end of the window: the next R-peak, on time;
- in **S**, the part before that spike looks much like an N; where does the next spike
  fall?
- in **V**, the beat itself is wide and has a shape of its own.

Then write down a strategy in one sentence. Is S a shape of its own that a model can
learn, or is it mostly *not N and not V*? If it is the second, what in a design could say
"unlike a normal beat"? Keep the sentence: 2.1.5 shows where S stands, and lesson 3 comes
back to it with a router that sends each input to one of several experts (goal 3.4).

### The metric

N is about 89% of the validation beats. A model that always says N scores 89% accuracy
and finds no patient (slide 22). These notebooks read the **macro recall**: the mean
of the three recalls, so a missed S counts as much as a missed N.

## 2.1.1 Baselines, before any kernel

Three floors, in lesson 1's order (goal 1.13):

- **always N**: the macro recall is exactly 1/3, whatever the data;
- **1-nearest-neighbour on the raw beat**: lesson 1 compared an encoder's vectors by
  cosine similarity. Here there is no encoder, so the raw beat of 187 steps is its own
  vector (goal 2.9), compared the same way: is a beat of an unseen patient closest to a
  beat of the same class?
- **a linear layer on `(B, 187)`**, and the same with one hidden layer: lesson 1's
  head, on the raw steps.

**Before you run the next cell**, write down what you expect 1-nearest-neighbour to
reach: above or below the linear layer, and why?

In [ ]:
majority = np.zeros(len(val), dtype=int)  # the index of N
print(
    f"{'always N':34s} macro recall {hb.scores(val.labels.numpy(), majority)['macro_recall']:.3f}"
)

knn = KNeighborsClassifier(n_neighbors=1, metric="cosine").fit(
    fit.beats[:, :, 0], fit.labels
)
nn1 = hb.scores(val.labels.numpy(), knn.predict(val.beats[:, :, 0]))
print(
    f"{'1-nearest-neighbour, cosine':34s} macro recall {nn1['macro_recall']:.3f}",
    {c: round(r, 2) for c, r in nn1["recall"].items()},
)


class Flat(nn.Module):
    """Lesson 1's head on the raw steps: `(B, S, 1) -> (B, S) -> (B, K)`."""

    def __init__(self, hidden: int = 0):
        super().__init__()
        layers = (
            [nn.Linear(hb.STEPS, hidden), nn.GELU(), nn.Linear(hidden, len(hb.CLASSES))]
            if hidden
            else [nn.Linear(hb.STEPS, len(hb.CLASSES))]
        )
        self.head = nn.Sequential(*layers)

    def forward(self, x):
        return self.head(x[..., 0])


linear, linear_s = hb.spread(lambda: Flat(), fit, val, EPOCHS, REPEATS)
print(hb.line("linear, (B, 187)", linear, linear_s))
mlp, mlp_s = hb.spread(lambda: Flat(hidden=256), fit, val, EPOCHS, REPEATS)
print(hb.line("one hidden layer of 256", mlp, mlp_s, versus=linear))
print(
    f"\n=> a later result must clear {mlp.mean() + 2 * mlp.std():.3f} to beat the floor"
)

Compare the numbers with what you wrote down. Without any training, the nearest raw beat
of another patient, by cosine, is a better guess than lesson 1's head with a hidden
layer: the raw beats already have a geometry of their own, and cosine reads it. Read the recall per class: nearly
every N, about six in ten V, and almost no S. That a normal beat is found by its cosine
alone is information for a design: a similarity to normal beats could be a feature of its
own. Whether it helps the class that matters, S, is a question to test, not to assume.

The floor every design below is read against is the hidden layer, in sigmas of the
larger of the two spreads; 1-nearest-neighbour has no spread, since it does not train.
Keep its number beside every design as well: a design that trains and does not clear it
has learned less than the geometry of the raw beats already gave.

## 2.1.2 What a kernel does

A kernel is a handful of weights that slides along an ordered axis. At every position it
multiplies the values under it and sums them; the sum is one value of the output. The
**same** weights are used at every position (slide 5).

`Conv1D(in_size, out_size, kernel_size)` takes `(B, S, D)` and gives `(B, S', D')`:
`D'` is the number of kernels, each one output channel. Below, kernels set by hand
rather than learned, on one beat: two that smooth, and two that answer where the beat
changes. The last two share one `Conv1D`, so they come out as two channels.

In [ ]:
def by_hand(*weights: list[float]) -> Conv1D:
    """One kernel per row of `weights`, set by hand, no bias: `(B, S, 1) -> (B, S, rows)`."""
    size = len(weights[0])
    conv = Conv1D(in_size=1, out_size=len(weights), kernel_size=size, padding=size // 2)
    with torch.no_grad():
        conv.conv.weight[:] = torch.tensor(weights).unsqueeze(1)
        conv.conv.bias.zero_()
    return conv


beat = fit.beats[np.flatnonzero(fit.labels.numpy() == 2)[0]]  # a V beat, (187, 1)
x = beat.unsqueeze(0)  # (1, 187, 1)
mean3 = by_hand([1 / 3] * 3)(x)
mean15 = by_hand([1 / 15] * 15)(x)
change = by_hand([-1.0, 0.0, 1.0], [1.0, -2.0, 1.0])(x)
print(f"two kernels in one Conv1D: in {tuple(x.shape)} -> out {tuple(change.shape)}")

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(t, beat[:, 0], color="0.6", lw=3, label="original")
axes[0].plot(t, mean3[0, :, 0].detach(), label="mean of 3")
axes[0].plot(t, mean15[0, :, 0].detach(), label="mean of 15")
axes[0].set_title("Smoothing: the mean over 3 and over 15 neighbours")
axes[1].axhline(0, color="0.85", lw=1)
axes[1].plot(t, change[0, :, 0].detach(), label="edge [-1, 0, 1]")
axes[1].plot(t, change[0, :, 1].detach(), label="curvature [1, -2, 1]")
axes[1].set_title("Change: where the beat rises or falls, and where it bends")
axes[1].set_xlabel("seconds after the R-peak")
for ax in axes:
    ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

The mean smooths. Over 3 neighbours it barely changes the beat; over 15, 0.12 s and
about as long as a QRS complex, the sharp R-peak flattens into a hump. The edge kernel is
positive where the beat climbs and negative where it drops. Its mirror `[1, 0, -1]` draws
the same line upside down, so a layer that learns either one has learned the same thing.
The curvature kernel `[1, -2, 1]` is the edge of the edge: it answers where the slope
itself changes. On this smooth beat it stays small, except at the sharp R-peak at the
start. A trained layer learns its own weights,
but every one of them does this: it looks at a few neighbours, with the same weights
everywhere.

### Why so few weights

A dense layer from 187 steps to 187 steps has a weight for every pair of positions. A
kernel of 3 has 3, reused at every position, and it only looks at neighbours (slide 3):

In [ ]:
dense = nn.Linear(hb.STEPS, hb.STEPS)
kernel = Conv1D(1, 1, kernel_size=3, padding=1)
print(f"dense 187 -> 187: {hb.n_params(dense):6d} weights")
print(
    f"kernel of 3:      {hb.n_params(kernel):6d} weights (3 and a bias), same output shape"
)

### The same operation at another rank

On an image `(B, C, H, W)` nothing new happens: the window is 3 by 3 and slides along
two axes instead of one (slide 6). `Conv2D` takes the channels on axis 1, as images in
this course are stored; `Conv1D` takes them last, as sequences are.

In [ ]:
image = torch.zeros(1, 1, 9, 9)
image[0, 0, 4, 1:8] = 1.0  # a cross
image[0, 0, 1:8, 4] = 1.0
conv2 = Conv2D(in_size=1, out_size=2, kernel_size=3, padding=1)
with torch.no_grad():
    edge = torch.tensor([[-1.0, 0.0, 1.0]] * 3)
    conv2.conv.weight[:] = torch.stack([edge, edge.T]).unsqueeze(1)
    conv2.conv.bias.zero_()
maps = conv2(image)
print(
    f"in {tuple(image.shape)} -> out {tuple(maps.shape)}: 9 weights per kernel, two kernels"
)

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, img, title in zip(
    axes,
    [image[0, 0], maps[0, 0], maps[0, 1]],
    ["input", "vertical edges", "horizontal edges"],
    strict=True,
):
    ax.imshow(img.detach(), cmap="RdBu_r", vmin=-3, vmax=3)
    ax.set_title(title)
    ax.axis("off")
plt.show()

### Two kernels without an activation are one kernel

Lesson 1 (notebook 1.2, goal 1.6) showed that a stack of linear layers without a
nonlinearity is one linear layer. A kernel is linear too, so the same holds: a kernel of
3 on a kernel of 5 is exactly one kernel of 7. The check below builds that one kernel
from the two and compares the outputs.

In [ ]:
k3, k5, k7 = Conv1D(1, 1, 3), Conv1D(1, 1, 5), Conv1D(1, 1, 7)
with torch.no_grad():
    k3.conv.bias.zero_()
    k5.conv.bias.zero_()
    # a kernel of 5, then one of 3, is one kernel whose weights are the two rows convolved
    w7 = np.convolve(k5.conv.weight[0, 0].numpy(), k3.conv.weight[0, 0].numpy())
    k7.conv.weight[0, 0] = torch.tensor(w7.copy())
    k7.conv.bias.zero_()
x = fit.beats[:4]
stacked, single = k3(k5(x)), k7(x)
print(
    f"shapes {tuple(stacked.shape)} and {tuple(single.shape)}, largest difference {(stacked - single).abs().max():.1e}"
)

Depth only buys something with an activation between the kernels. Every stack below
puts a GELU after each kernel.

## 2.1.3 Fewer positions on an axis

Downsampling is nothing more than fewer positions on an axis (slide 7). Three ways:

- a kernel with **stride 2** jumps two positions each time; its weights decide how two
  positions become one: learned;
- a **pool** of 2 keeps the larger (max) or the mean of each pair: a fixed rule, no
  weights;
- a **global pool** takes the whole axis into one value per channel: an aggregation,
  the axis is gone. Lesson 1's mean over chunks was one.

In [ ]:
x = fit.beats[:8]  # (B, S, D) = (8, 187, 1)
x16 = Conv1D(1, 16, 5, padding=2)(x)
print(f"input                       {tuple(x.shape)}")
print(f"kernel of 5, 16 channels    {tuple(x16.shape)}")
print(
    f"kernel of 5 with stride 2   {tuple(Conv1D(16, 16, 5, stride=2, padding=2)(x16).shape)}"
)
print(f"max pool of 2               {tuple(Pool1D(2)(x16).shape)}")
print(
    f"global mean pool            {tuple(x16.mean(dim=1).shape)}   <- the S axis is gone"
)

## 2.1.4 How far one position sees

The **receptive field** of an output position is the number of input steps it depends
on. A kernel of 5 sees 5. Stack two and the second sees 5 positions of the first, each
of which saw 5: 9 in all. A pool of 2 between them doubles the step size for every layer
above, so the field grows much faster (slide 8).

`stack(depth)` below builds `depth` blocks. Every block is a kernel of 5 and a GELU,
with or without a max pool of 2 after it; the kernel size is the argument
`kernel_size`. Without pools every kernel of 5 adds 4 steps, so `depth` kernels see
`1 + 4 * depth`. With pools the step a kernel takes on the input doubles after every
block.

**Before you run the next cell**: a beat is 187 steps. How many kernels of 5 do you need
before one position sees the whole beat, without pools, and with a pool of 2 after every
kernel?

In [ ]:
def stack(
    depth: int, width: int = 32, kernel_size: int = 5, pool: bool = True
) -> list[nn.Module]:
    """`depth` blocks of a kernel and a GELU, each followed by a max pool of 2 when `pool`."""
    layers, channels = [], 1
    for _ in range(depth):
        layers += [
            Conv1D(channels, width, kernel_size, padding=kernel_size // 2),
            nn.GELU(),
        ]
        if pool:
            layers.append(Pool1D(2))
        channels = width
    return layers


print("every block: a kernel of 5 and a GELU; with pools, a max pool of 2 after it")
print(f"{'depth':>5} {'field, no pools':>16} {'field, pool of 2':>17}")
for depth in range(1, 7):
    print(
        f"{depth:5d} {hb.receptive_field(stack(depth, pool=False)):16d} {hb.receptive_field(stack(depth)):17d}"
    )

The table is arithmetic. The figure below puts it on a beat, with the V beat of 2.1.2.
Take one output position of the stack with pools, the one whose field is centred at
0.4 s, and ask which input steps can move it. The top panel shades that stretch for
depth 1, 3 and 5. The bottom panel measures it for an untrained stack: the gradient of
that one output with respect to every input step, averaged over random inputs. Outside
the field it is zero, whatever the weights. Inside, it fades towards the edges, because
fewer paths through the layers lead from an edge step to the output.

Read the top panel against the beat: depth 1 sees a sliver of one wave, depth 3 a whole
wave, depth 5 more than the beat. Its field of 156 steps runs off the start, so the
shaded part is shorter than the field.

In [ ]:
CENTRE = 50  # input step 50, 0.4 s after the R-peak
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(t, beat[:, 0], color="0.3", lw=1.5, label="the V beat")
for colour, depth in zip(("C0", "C1", "C2"), (1, 3, 5), strict=True):
    features = nn.Sequential(*stack(depth))
    field = hb.receptive_field(list(features))
    grad = hb.reach(features, CENTRE)
    seen = t[grad > 0]
    label = f"depth {depth}: {field} steps, {field / 125:.2f} s"
    axes[0].axvspan(seen.min(), seen.max(), color=colour, alpha=0.15, label=label)
    axes[1].plot(t, grad / grad.max(), color=colour, label=label)
axes[0].set_title("What one output position, centred at 0.4 s, sees of the beat")
axes[1].set_title("How strongly each input step can move that output (untrained)")
axes[1].set_ylabel("relative gradient")
axes[1].set_xlabel("seconds after the R-peak")
for ax in axes:
    ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 2.1.5 Does a wider view pay?

Now the bet itself. A kernel bets on **locality**: a pattern means the same wherever it
occurs (slide 18, goal 2.7). A V beat is wide and odd-shaped over its whole length; to
see that, one position may need to see most of the beat.

The models below are the same stack at depth 1, 3 and 5, with a pool of 2 after every
kernel, then flattened into a linear layer to the three classes. Depth changes the
receptive field, and the number of weights with it; the table prints both.

**Before you run the next cell**, write down what you expect: does the macro recall
keep rising with depth, flatten, or fall? And where would you expect S to stand?

In [ ]:
class Kernels(nn.Module):
    """A stack of kernels, flattened into a linear head: `(B, S, 1) -> (B, K)`."""

    def __init__(self, depth: int, width: int = 32, kernel_size: int = 5):
        super().__init__()
        self.features = nn.Sequential(*stack(depth, width, kernel_size))
        with torch.no_grad():
            flat = self.features(torch.zeros(1, hb.STEPS, 1)).flatten(1).shape[1]
        self.head = nn.Linear(flat, len(hb.CLASSES))

    def forward(self, x):
        return self.head(self.features(x).flatten(1))


depths, depth_seconds = {}, {}
for depth in (1, 3, 5):
    values, seconds = hb.spread(lambda d=depth: Kernels(d), fit, val, EPOCHS, REPEATS)
    depths[depth], depth_seconds[depth] = values, seconds
    field = hb.receptive_field(list(Kernels(depth).features))
    print(
        hb.line(
            f"depth {depth}, field {field:3d}, {hb.n_params(Kernels(depth)):6d} weights",
            values,
            seconds,
            versus=mlp,
        )
    )

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
fields = [hb.receptive_field(list(Kernels(d).features)) for d in depths]
means = [v.mean() for v in depths.values()]
stds = [v.std() for v in depths.values()]
ax.errorbar(fields, means, yerr=stds, marker="o", capsize=4, label="kernel stacks")
ax.axhline(mlp.mean(), color="0.5", ls="--", label="one hidden layer, no kernels")
for f, d, m in zip(fields, depths, means, strict=True):
    ax.annotate(f"depth {d}", (f, m), textcoords="offset points", xytext=(6, -12))
ax.set_xlabel("receptive field, in steps")
ax.set_ylabel("macro recall over N, S, V")
ax.set_title("Does a wider view pay on unseen patients?")
ax.legend()
plt.show()

Read it in sigmas against the floor, as in lesson 1. Two things to look at:

- **the trend**: compare it with what you wrote down. From depth 1 to depth 3 the
  number of weights stays about the same while the field grows from 6 steps to 36, so
  what changed is mostly the view. From depth 3 to 5 both the field and the weights grow:
  is that step still more than two sigmas? A design that separates field from weights:
  one kernel of 37 at depth 1 sees as far as depth 3 does.
- **S**: print the recall per class of one model (`hb.train(Kernels(5), fit, val,
  EPOCHS)`). An S beat starts too early. Its shape looks much like an N; what is odd is
  mostly *when* it comes, relative to the beats around it, and a cut-out beat holds only
  a trace of that. That is a limit of the bet on locality, and where notebook 2.2
  starts.

- **a bet from first principles**: a QRS complex lasts about 0.1 s, 12 to 15 steps at
  125 Hz. Slide 8 draws it: a kernel of 15 sees the whole complex in one layer, where a
  kernel of 5 needs depth to get there. That is a hypothesis worth a study of its own.
  `kernel_size` is in `SPACE` in `exercises/study.py` with 3, 5 and 9: add 15, say what
  you expect, and test it; `parallel` puts a wider kernel beside the first. It is also a
  question the class can tune together, each building on the other's runs.

## 2.1.6 The same beat as a 2D grid

187 is 11 times 17. Fold the beat row by row into a grid `(B, 1, 11, 17)` and a 3-by-3
kernel sees, besides the steps before and after, the steps one row above and below:
17 steps apart (slide 19; slide 4 shows this beat). The data does not change; the shape does, and with it which
neighbours a kernel sees (goal 2.8). It is a bet that what happens 17 steps later belongs
with what happens now.

In [ ]:
grid = beat[:, 0].reshape(11, 17)
fig, axes = plt.subplots(1, 2, figsize=(12, 3), gridspec_kw={"width_ratios": [2, 1]})
axes[0].plot(beat[:, 0])
for row in range(1, 11):
    axes[0].axvline(row * 17, color="0.85", lw=0.8)
axes[0].set_title("(187, 1): the rows of the fold marked")
axes[1].imshow(grid, cmap="viridis", aspect="auto")
axes[1].set_title("(1, 11, 17): the same 187 values")
plt.show()


class Folded(nn.Module):
    """The beat folded to `(B, 1, 11, 17)`, two 2D kernels with a pool, a linear head."""

    def __init__(self, width: int = 32):
        super().__init__()
        self.features = nn.Sequential(
            Conv2D(1, width, 3, padding=1),
            nn.GELU(),
            Pool2D(2),
            Conv2D(width, width, 3, padding=1),
            nn.GELU(),
        )
        with torch.no_grad():
            flat = self.features(torch.zeros(1, 1, 11, 17)).flatten(1).shape[1]
        self.head = nn.Linear(flat, len(hb.CLASSES))

    def forward(self, x):
        grid = x[..., 0].reshape(-1, 1, 11, 17)  # (B, 187, 1) -> (B, 1, 11, 17)
        return self.head(self.features(grid).flatten(1))


print(
    f"folded: {hb.n_params(Folded()):6d} weights;  depth 3 in 1D: {hb.n_params(Kernels(3)):6d} weights"
)

**Before you run the next cell**, write down what you expect from the fold against the
1D stack of depth 3, and why. Then run both arms, each repeated.

In [ ]:
folded, folded_s = hb.spread(Folded, fit, val, EPOCHS, REPEATS)
print(hb.line("depth 3 in 1D", depths[3], depth_seconds[3]))
print(hb.line("folded to (11, 17)", folded, folded_s, versus=depths[3]))

Inside about two sigmas, the difference is no result, whatever its sign. A beat has no
period of 17 steps: the fold relates samples that have nothing to do with each other, and
its kernels have to learn to ignore the row above. A fold pays where the data repeats
with the period you fold by: a day of ten-minute readings folded by 144, a year by 12.
You choose the shape; the data does not tell the kernel it was folded.

## What you now know

- A kernel scans locally along an ordered axis with the same weights everywhere; on
  `(B, S, D)` and on `(B, C, H, W)` it is one operation, and it needs far fewer weights
  than a dense layer.
- Stacked kernels without an activation are one kernel.
- Fewer positions: a stride learns how, a pool follows a rule, a global pool removes
  the axis.
- The receptive field grows with depth, and much faster with pools; it is a property of
  the design.
- A kernel bets on locality. On these beats a wider view paid from a field of 6 steps
  to one of 36; the class every design misses most, S, is the one whose oddness lies
  mostly outside the beat.
- Folding a series into a grid is a design choice that changes which neighbours a kernel
  sees; it pays only when the data repeats with the fold.